# Relevance-atom judge — Phase 1 pilot

**Objective.** Drive the `relevance_judge` package one gate at a time and inspect the
results as live objects. The judge asks a single question per (query, doc) — *"is this
document relevant to this query?"* — and never sees route names, lists, or scores. New
judged pairs extend the qrels; route scores recompute arithmetically at scoring time.

**Done** = the accuracy gate passes on positive precision, the sub-1.0-tie pilot has judged
its ~396 above-gold pairs, and `PilotScorer` reports the qrels-hole rate + per-route
discovered-relevance.

Plan: `~/.claude/plans/can-you-use-the-shimmering-hare.md` · doc:
`docs/research/relevance-judge-recovery.md` · CLI twin: `src/scripts/run_relevance_judge.py`.

In [1]:
# Setup: imports, config, and a spend guard (nothing here calls an LLM)
from __future__ import annotations

import pandas as pd

from augmentation.engine import Budget
from relevance_judge import (
    JudgeQueue,
    JudgeRunLog,
    PilotScorer,
    RelevanceJudge,
    RelevanceJudgeConfig,
    Sources,
    ValidationHarness,
)

pd.set_option("display.max_colwidth", 90)

SEED = 0
SPEND = False          # flip to True to make real LLM calls (needs OPENROUTER_API_KEY + gpt-5.6-luna)
MAX_SPEND_USD = 5.0    # hard ceiling enforced per-call by Budget

config = RelevanceJudgeConfig()


def budget() -> Budget:
    return Budget(
        MAX_SPEND_USD,
        usd_per_mtok_in=config.engine.usd_per_mtok_in,
        usd_per_mtok_out=config.engine.usd_per_mtok_out,
    )


config.engine.model, config.artifacts

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('openrouter/openai/gpt-5.6-luna',
 PosixPath('/Users/andrei/projects/hybrid-search-rrf-dataset/src/data/relevance_judge'))

## The gate ladder

Each stage gates the next; the free ones cannot be invalidated by the open SPEC decision.

| stage | spend | what it decides |
|---|---|---|
| §1.0 precondition audit | free | how many residual rows can be rescored at all |
| §1.1 accuracy gate | ~$1 | **positive precision** — is the judge safe to trust as gold? |
| §1.2 sub-1.0-tie pilot | ~$0.12 | judge above-gold docs; break ties arithmetically |
| score | free | qrels-hole rate · per-route discovered-relevance · tie-conversion |

Merge rule: judged atoms carry `source="llm"`; `QrelStore` precedence means a human
judgment always wins a conflict, so human truth stays authoritative.

## §1.0 — Precondition audit (free)

Reproduces the arch5k residual: `all_tied` ∪ `absent@500`, and how many have persisted `route_rankings` to rescore against.

In [2]:
queue = JudgeQueue(config)
audit = queue.precondition_audit()
pd.Series(audit, name="residual population")

residual         2045
all_tied         1917
sub1_ties          85
absent@500        128
with_rankings    2045
no_rankings         0
Name: residual population, dtype: int64

## §1.1 — Accuracy gate (spends ~$1 when `SPEND=True`)

The **hard gate is precision + a recall floor**, not agreement: a judged atom becomes gold, so a
false positive injects wrong truth (the dangerous error); a conservative judge that answers
strictly is *safe* for gold injection, so overall/per-lane agreement are reported as diagnostics,
not enforced. Stages are inspectable — `sample()` → `judge_rows()` → `score()` — a per-lane
sampling table, a live progress bar, per-pair predictions in `validation_predictions.parquet`,
a confusion breakdown, and a false-positive audit.

**Two caveats this surfaces.** (1) Precision is only measurable where humans judged *irrelevant*
docs; residual cover lanes are positive-only, so the gate borrows negatives from graded lanes
(non-human/synthetic lanes like `augmentation` are excluded automatically). (2) Some datasets
grade *topical/partial* matches relevant while the judge answers strict *"does this answer the
query"* — so low recall there is a definition gap, not a judge failure. Always audit the false
positives (last cell): a grade-0 doc the judge calls relevant may be a genuine qrels-hole.

In [3]:
# The judge's instruction — the precision lever. To iterate: edit
# relevance_judge/judge.py::INSTRUCTION, restart the kernel, re-run --validate.
from relevance_judge.judge import INSTRUCTION
print(INSTRUCTION)

You judge whether a document contains the answer a search query needs — not whether it shares the query's topic.
Say yes only if the document contains the specific fact, entity, value, procedure, fix, ruling, argument, or explanation the query asks for — either fully, or as a substantial part that genuinely helps answer it. A fact equivalent in substance counts (an annulment answers whether a couple divorced). Say no if that answer content is absent, even when the document is closely related: same subject, same product or API, a different aspect, background, a setup or framing that never answers, or the query's terms repeated. Shared words and topical closeness are not evidence of relevance. A blank, boilerplate, or title-only document is never relevant. When you cannot confirm the answer content is actually present, say no — a wrong yes becomes permanent gold that corrupts every score built on it, while a wrong no costs only one missed pair.
Your reason must name the specific answer c

In [4]:
# Lane selection (free): where the judge is applied vs where precision is measurable
harness = ValidationHarness(config)
residual_lanes = sorted(queue.residual()["dataset"].unique())
negative_lanes = Sources(config).lanes_with_negatives()
val_lanes = sorted(set(residual_lanes) | set(negative_lanes))

print("residual lanes (judge applied here):", residual_lanes)
print("lanes WITH judged negatives (precision measurable):", negative_lanes)
print("residual lanes WITHOUT negatives (recall-only):",
      sorted(set(residual_lanes) - set(negative_lanes)))

residual lanes (judge applied here): ['clerc', 'crumb-legal-qa', 'finder', 'quest', 'rarb-math', 'scirgen-geo-en']
lanes WITH judged negatives (precision measurable): ['beir-touche-2020', 'crumb-clinical-trial', 'crumb-paper-retrieval', 'crumb-set-operation-entity-retrieval', 'dbpedia-entity', 'freshstack-angular', 'freshstack-godot', 'freshstack-langchain', 'freshstack-laravel', 'freshstack-yolo', 'miracl-en-dev', 'wands']
residual lanes WITHOUT negatives (recall-only): ['clerc', 'crumb-legal-qa', 'finder', 'quest', 'rarb-math', 'scirgen-geo-en']


In [5]:
# Stage 1 — sample() is FREE: builds the eval frame and prints a per-lane pos/neg/dropped table
eval_rows = harness.sample(val_lanes, per_lane=200, seed=SEED)
eval_rows.groupby(["dataset", "human_relevant"]).size().unstack(fill_value=0)

sampling per lane:
                                lane  pos  neg  kept  dropped_no_text
                    beir-touche-2020  100  100   200                0
                               clerc  200    0   200                0
                crumb-clinical-trial  100  100   200                0
                      crumb-legal-qa  200    0   200                0
               crumb-paper-retrieval  100  100   200                0
crumb-set-operation-entity-retrieval  165   35   200                0
                      dbpedia-entity  100  100   200                0
                              finder  200    0   200                0
                  freshstack-angular  100  100   200                0
                    freshstack-godot  100  100   200                0
                freshstack-langchain  100  100   200                0
                  freshstack-laravel  100  100   200                0
                     freshstack-yolo  100  100   200                0
 

human_relevant,False,True
dataset,,
beir-touche-2020,100,100
clerc,0,200
crumb-clinical-trial,100,100
crumb-legal-qa,0,200
crumb-paper-retrieval,100,100
crumb-set-operation-entity-retrieval,35,165
dbpedia-entity,100,100
finder,0,200
freshstack-angular,100,100


In [6]:
# Stage 2 — judge_rows() shows a live tqdm bar and banks validation_predictions.parquet
# every 50 rows (crash-safe). It reuses eval_rows from Stage 1 (no re-sampling) and returns
# the per-pair predictions as an inspectable frame.
if SPEND:
    preds = harness.judge_rows(eval_rows, budget=budget())
    preds.head()
else:
    preds = None
    print(f"set SPEND=True to judge {len(eval_rows)} validation pairs (tqdm bar appears here)")

set SPEND=True to judge 3600 validation pairs (tqdm bar appears here)


In [7]:
# Stage 3 — score() is pure (no LLM); finalize() writes the report + opens a judge run on pass
if preds is not None:
    report = harness.finalize(harness.score(preds, val_lanes))
    print("GATE:", "PASS" if report["passed"] else "FAIL", "->", config.validation_report)
    report_view = {k: v for k, v in report.items() if k != "agreement_by_lane"}
else:
    report_view = {"skipped": "run Stage 2 with SPEND=True first"}

report_view

{'skipped': 'run Stage 2 with SPEND=True first'}

### Re-score without spending

The judgments are banked in `validation_predictions.parquet`; scoring is pure arithmetic. So after
any gate change (thresholds, referee rules) you re-derive the verdict for **free** — no re-judging.
This is the notebook twin of `run_relevance_judge.py --validate --rescore`.

In [8]:
# Re-score the banked predictions under the CURRENT gate logic — NO LLM spend (read-only here).
# To also open a judge run (unlocks the pilot), use the CLI: `--validate --rescore`.
if config.validation_predictions.exists():
    banked = pd.read_parquet(config.validation_predictions)
    valid = set(Sources(config).lanes_with_negatives()) | set(config.anchor_lanes)
    # drop lanes that only LOOK negative under a stale threshold (e.g. nfcorpus grade-1)
    keep = banked.groupby("dataset").filter(
        lambda g: g.name in valid or not bool((~g["human_relevant"]).any())
    )
    rescored = harness.score(keep, sorted(keep["dataset"].unique()))
    print("GATE (rescored, no spend):", "PASS" if rescored["passed"] else "FAIL")
    rescored_view = {k: rescored[k] for k in (
        "passed", "precision_relevant", "anchor_recall", "recall_relevant",
        "false_positives", "n_validation")}
else:
    rescored_view = {"skipped": "no banked predictions yet — run Stage 2 once"}
rescored_view

GATE (rescored, no spend): PASS


{'passed': True,
 'precision_relevant': 0.955026455026455,
 'anchor_recall': 0.9112903225806451,
 'recall_relevant': 0.45527584359935724,
 'false_positives': 17,
 'n_validation': 2959}

In [9]:
# Audit the false positives — the whole reason precision is the gate. audit_false_positives()
# enriches each FP with the human grade + query/doc text and banks false_positives_audit.parquet.
# Hand-check each: a genuine judge error, or a qrels-hole (relevant-but-unjudged) the program exists
# to find? (A grade-0 doc the judge calls relevant may be a real hole, not a mistake.)
if config.validation_predictions.exists():
    fp_audit = harness.audit_false_positives(per_lane=10)
    print(f"{len(fp_audit)} false positives; by lane:")
    print(fp_audit["dataset"].value_counts().to_string())
    display(fp_audit[["dataset", "human_grade", "judge_reason", "query", "doc"]].head(15))
else:
    print("no predictions yet — run Stage 2 with SPEND=True")

30 false positives; by lane:
dataset
beir-nfcorpus           10
freshstack-langchain     6
freshstack-yolo          5
antique                  3
freshstack-angular       2
freshstack-godot         2
beir-touche-2020         1
wands                    1


,dataset,human_grade,judge_reason,query,doc
0,antique,2,recommends Compound W or duct-tape occlusion to remove a hand wart at home,How do you get rid of wart on your hand a using home method?,"First of all, do not ever bite a wart. Warts are viruses, you can spread a virus. If y..."
1,antique,2,"It attributes anti-Islam sentiment to perceived intolerance, terrorism, violence, and ...",why is islam so hated in america?,Then I guess you Muslims shouldn't be attacking non-Muslims. Islam is a religion of in...
2,antique,2,recommends a calorie deficit through proper diet and cardiovascular exercise to lose fat,"How to ""LOSE""all body fat ?","Have you ever seen body builders in their off-season....all big bulky back and arms, h..."
3,beir-nfcorpus,1,"describes the Physicians’ Health Study II trial, its design, participants, interventio...",Harvard Physicians’ Study II,Multivitamins in the Prevention of Cancer in Men: The Physicians’ Health Study II Rand...
4,beir-nfcorpus,1,discusses consumer-perceived risks of pesticide residues in conventional produce.,pesticides,"Perceived risks of conventional and organic produce: pesticides, pathogens, and natura..."
5,beir-nfcorpus,1,"discusses chronic liver disease, its causes, mechanisms, and progression to hepatocell...",liver disease,Obesity-associated mechanisms of hepatocarcinogenesis.\n\nObesity has been recognized ...
6,beir-nfcorpus,1,identifies trans-fatty acids as low in ancestral plant-based diets,trans fats,"The Garden of Eden--plant based diets, the genetic drive to conserve cholesterol and i..."
7,beir-nfcorpus,1,It compares very-low-carbohydrate ketogenic diets with low-fat diets and reports weigh...,low-carb diets,Very-low-carbohydrate ketogenic diet v. low-fat diet for long-term weight loss: a meta...
8,beir-nfcorpus,1,discusses pig-meat consumption and its association with chronic liver disease mortality,pork,National mortality rates from chronic liver disease and consumption of alcohol and pig...
9,beir-nfcorpus,1,Document discusses neurocysticercosis cases and intracerebral lesion evaluation.,neurocysticercosis,Seroprevalence of cysticercosis in an Orthodox Jewish community.\n\nNeurocysticercosis...


## §1.2 — Sub-1.0-tie pilot

The work list is free to build (below) — above-gold docs only, the only docs whose relevance
can break a sub-1.0 tie. The drop accounting is printed: `no-rankings` / `gold-at-v2-rank-1`
(the l2-tie vs persisted-v2-ranking mismatch) / `no-text`.

In [10]:
pairs = queue.sub1_pairs()   # prints the drop accounting
pairs.head()

sub-1.0 ties: 85 rows; dropped 0 no-rankings, 15 gold-at-v2-rank-1, 0 no-text; 396 pairs to judge


,dataset,query_id,doc_id,query,doc_text
0,clerc,382396,83621,"28 U.S.C. § 1391, and the Southern District of New York (the transferee court), see 28...",defendant that is a corporation shall be deemed to reside in any judicial district in ...
1,clerc,205733,23164490,"hearing, the court, on its own motion and not at the request or suggestion of any part...","auspices of a Chapter 13 plan. See also, In re Edwards, 50 B.R. 933 (Bkcy.S.D.N.Y.1985..."
2,clerc,568352,7822974,is common to the class. Common issues thus will predominate in plaintiffs’ fraud claim...,"to disclose the same material information to the class. See, e.g., In re Checking Acco..."
3,clerc,236845,8059411,his testimony given at a hearing on a motion to suppress. The precise holding in Simmo...,"testify, not under compulsion from the Court, but in order to establish standing to ob..."
4,clerc,737704,11200556,of Defendants’ representations that they had fully complied with the Court’s order of ...,"motion] and how they are to be obtained, (2) how those facts are reasonably expected t..."


In [11]:
# Judge the pairs — needs SPEND=True and a passed §1.1 run
runs = JudgeRunLog(config).load()
passed = runs[runs["passed"]] if not runs.empty else runs
run_id = str(passed.iloc[-1]["judge_run_id"]) if not passed.empty else None

if SPEND and run_id:
    counts = RelevanceJudge(config).judge_pairs(pairs, run_id=run_id, budget=budget())
elif SPEND:
    counts = {"error": "no passed validation run — run §1.1 first"}
else:
    counts = {"skipped": f"set SPEND=True (after §1.1) to judge {len(pairs)} pairs (~$0.12)"}

counts

{'skipped': 'set SPEND=True (after §1.1) to judge 396 pairs (~$0.12)'}

## Results — what the atoms bought

Returns `{'error': 'no judged atoms yet'}` until §1.2 has run. `qrels_hole_rate` is the headline (stack-independent); `discovered_relevance_by_route` is the sparse blind-spot audit; `tie_conversion_rate` uses the persisted v2 rankings (a stated approximation for l2-defined ties).

In [12]:
result = PilotScorer(config).audit()
result

{'atoms': 371,
 'rows_judged': 66,
 'qrels_hole_rate_pairs': 0.1509433962264151,
 'qrels_hole_rate_rows': 0.5151515151515151,
 'discovered_relevance_by_route': {'dense_only': {'judged': 332,
   'relevant': 46,
   'discovered_relevance_rate': 0.13855421686746988},
  'sparse_only': {'judged': 136,
   'relevant': 49,
   'discovered_relevance_rate': 0.3602941176470588},
  'pure_rrf': {'judged': 274,
   'relevant': 55,
   'discovered_relevance_rate': 0.20072992700729927}},
 'tie_conversion_rate': 0.0,
 'regime_after_counts': {'low_margin': 46,
  'all_tied': 16,
  'decisive_strong': 4}}

## §1.2b — True tie-conversion via re-derived l2 rankings

`tie_conversion` above read 0 because it scored the persisted **v2** rankings, while the ties are
defined on the stronger **l2 (gemini)** stack whose ranked lists were never saved (only the scores
were). The gemini **collections still exist**, and the documents are already embedded in them — so
re-deriving the l2 rankings is **query-side only**: embed each pilot query with gemini, search, and
score before/after adding the judged gold. The query is embedded **server-side** by Qdrant's managed
inference, so this must hit the **cloud** instance that did the indexing: `QDRANT_CLOUD_URL` +
`QDRANT_CLOUD_API_KEY` + `cloud_inference=True` (same as `architecture_5k_test.ipynb`), and
`OPEN_ROUTER_API_KEY` for the embedding call.

In [13]:
# Re-derive l2 (gemini) rankings from the live collections — query-side only (~$0.001, no re-index).
import os
from qdrant_client import QdrantClient

from scripts.legb import gemini_dense_cfg, SPARSE_CFG, LegBPilot
from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy, PureRRFStrategy, SparseOnlyStrategy,
)
from hybrid_search_rrf_dataset.objective import RouterObjective
from hybrid_search_rrf_dataset.qrels import QrelStore
from relevance_judge.queue import regime

# The gemini collections AND the managed inference live on the cloud instance — a plain
# QDRANT_URL has no InferenceService and errors with "InferenceService URL not configured".
QDRANT = bool(os.getenv("QDRANT_CLOUD_URL"))

if QDRANT:
    _client = QdrantClient(url=os.environ["QDRANT_CLOUD_URL"],
                           api_key=os.environ["QDRANT_CLOUD_API_KEY"],
                           timeout=120, cloud_inference=True)   # server-side query embedding
    _dense = gemini_dense_cfg()                                 # reads OPEN_ROUTER_API_KEY
    _pilot = LegBPilot(_client, _dense)                         # collection naming only
    _strats: dict[str, dict] = {}

    def l2_rankings(lane, query):
        coll = _pilot.collection(lane)
        if coll not in _strats:
            _strats[coll] = {
                "dense_only": DenseOnlyStrategy(_client, coll, _dense, SPARSE_CFG),
                "sparse_only": SparseOnlyStrategy(_client, coll, _dense, SPARSE_CFG),
                "pure_rrf": PureRRFStrategy(_client, coll, _dense, SPARSE_CFG),
            }
        return {r: st.rank(query) for r, st in _strats[coll].items()}

    print("ready — documents already embedded in the gemini collections; only queries are embedded")
else:
    print("set QDRANT_CLOUD_URL + QDRANT_CLOUD_API_KEY + OPEN_ROUTER_API_KEY to re-derive l2 rankings")

ready — documents already embedded in the gemini collections; only queries are embedded


In [14]:
# Sanity: the re-derived l2 dense/rrf score should reproduce the stored rows.json l2 (same rankings).
if QDRANT:
    import json
    draw = {(r["dataset"], str(r["query_id"])): r
            for r in json.loads((config.arch5k / "rows.json").read_text())}
    obj = RouterObjective(min_relevance=config.min_relevance)
    s = Sources(config)
    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str})
    check = []
    for ds, qid in list(atoms.groupby(["dataset", "query_id"]).groups)[:6]:
        row = draw.get((ds, qid))
        if row is None:
            continue
        q = s.query_text(ds, {qid}).get(qid, "")
        gold = {d: 1 for d in s.manifest_gold(ds).get(qid, set())}
        redo = {r: obj.assess(rank, gold)[0] for r, rank in l2_rankings(ds, q).items()}
        check.append({"dataset": ds, "query_id": qid,
                      "stored_dense": round(row["l2"]["dense_only"], 3),
                      "redo_dense": round(redo["dense_only"], 3),
                      "stored_rrf": round(row["l2"]["pure_rrf"], 3),
                      "redo_rrf": round(redo["pure_rrf"], 3)})
    display(pd.DataFrame(check))
else:
    print("skipped (no QDRANT_URL)")

,dataset,query_id,stored_dense,redo_dense,stored_rrf,redo_rrf
0,clerc,123300,0.129,0.129,0.129,0.129
1,clerc,130321,0.189,0.189,0.189,0.189
2,clerc,135523,0.189,0.189,0.189,0.189
3,clerc,205733,0.189,0.189,0.189,0.189
4,clerc,236845,0.189,0.189,0.189,0.189
5,clerc,316140,0.129,0.129,0.129,0.129


In [15]:
# True tie-conversion: re-retrieve l2 rankings for every judged row, score before (human gold)
# vs after (human + judged), and count all_tied rows that become decisive/low-margin.
if QDRANT:
    from tqdm.auto import tqdm
    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str, "relevance": int})
    datasets = sorted(atoms["dataset"].unique())
    human = PilotScorer(config)._human_store(datasets)
    merged = QrelStore.concat([human, RelevanceJudge(config).as_qrelstore()])
    before = {ds: human.lookup(ds) for ds in datasets}
    after = {ds: merged.lookup(ds) for ds in datasets}
    obj = RouterObjective(min_relevance=config.min_relevance)
    s = Sources(config)

    rows = []
    for ds, qid in tqdm(list(atoms.groupby(["dataset", "query_id"]).groups), desc="l2 re-retrieve"):
        q = s.query_text(ds, {qid}).get(qid, "")
        if not q:
            continue
        rk = l2_rankings(ds, q)
        sb = {r: obj.assess(rank, before[ds].get(qid, {}))[0] for r, rank in rk.items()}
        sa = {r: obj.assess(rank, after[ds].get(qid, {}))[0] for r, rank in rk.items()}
        rows.append({"dataset": ds, "query_id": qid,
                     "regime_before": regime(sb), "regime_after": regime(sa),
                     "resolved": regime(sb) == "all_tied" and regime(sa) != "all_tied"})
    l2conv = pd.DataFrame(rows)
    tied = int((l2conv["regime_before"] == "all_tied").sum())
    broke = int(l2conv["resolved"].sum())
    print(f"rows: {len(l2conv)} | all_tied under l2 before: {tied} | broke after judged gold: {broke}")
    print("l2 tie_conversion (of all_tied rows):", round(broke / tied, 3) if tied else "n/a")
    display(l2conv.groupby(["regime_before", "regime_after"]).size().rename("rows"))
else:
    print("skipped (no QDRANT_URL)")

l2 re-retrieve:   0%|          | 0/66 [00:00<?, ?it/s]

l2 re-retrieve:   2%|▏         | 1/66 [00:01<02:00,  1.85s/it]

l2 re-retrieve:   3%|▎         | 2/66 [00:03<02:01,  1.90s/it]

l2 re-retrieve:   5%|▍         | 3/66 [00:05<01:57,  1.87s/it]

l2 re-retrieve:   6%|▌         | 4/66 [00:07<01:53,  1.83s/it]

...

l2 re-retrieve:  98%|█████████▊| 65/66 [01:37<00:01,  1.75s/it]

l2 re-retrieve: 100%|██████████| 66/66 [01:39<00:00,  1.71s/it]

l2 re-retrieve: 100%|██████████| 66/66 [01:39<00:00,  1.50s/it]

rows: 49 | all_tied under l2 before: 48 | broke after judged gold: 5
l2 tie_conversion (of all_tied rows): 0.104


regime_before  regime_after   
all_tied       all_tied           43
               decisive_strong     1
               low_margin          4
low_margin     low_margin          1
Name: rows, dtype: int64

## §1.2c — Corrected pilot: judge the l2 above-gold docs

The §1.2 pilot picked above-gold docs from the **v2** rankings, but the ties are **l2**-defined — so
a judged doc can sit outside l2's top-k and never move the l2 score. Here we re-derive each row's
above-gold set **from the l2 rankings**, judge the ones we missed, and re-measure conversion on the
right doc set. (Bonus: this also covers the 15 rows dropped earlier for gold-at-v2-rank-1.)

In [16]:
# Fetch l2 rankings once for every sub-1.0 tie, derive l2 above-gold, and diagnose the v2/l2 gap.
if QDRANT:
    from tqdm.auto import tqdm
    s = Sources(config)
    sub1 = queue.residual()
    sub1 = sub1[(sub1["regime"] == "all_tied") & sub1["sub1"]].reset_index(drop=True)
    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str, "relevance": int})
    judged = set(zip(atoms["dataset"], atoms["query_id"], atoms["doc_id"]))

    l2_cache, l2_above = {}, {}
    for r in tqdm(list(sub1.itertuples(index=False)), desc="l2 above-gold"):
        gold = s.manifest_gold(r.dataset).get(r.query_id, set())
        rk = l2_rankings(r.dataset, r.query)
        above = set()
        for scores in rk.values():
            order = [d for d, _ in sorted(scores.items(), key=lambda kv: -kv[1])]
            cut = next((i for i, d in enumerate(order) if d in gold), len(order))
            above.update(order[:cut])
        l2_cache[(r.dataset, r.query_id)] = rk
        l2_above[(r.dataset, r.query_id)] = above - gold

    total_above = sum(len(a) for a in l2_above.values())
    already = sum(1 for (k, a) in l2_above.items() for d in a if (k[0], k[1], d) in judged)
    judged_in_l2 = sum(
        1 for (ds, qid, did) in judged
        if did in set().union(*(set(o) for o in l2_cache.get((ds, qid), {}).values()), set())
    )
    print(f"sub-1.0 ties: {len(sub1)} | l2 above-gold docs: {total_above} "
          f"(already judged: {already}, MISSED: {total_above - already})")
    print(f"of {len(judged)} judged pairs, only {judged_in_l2} land in some l2 top-k "
          f"-> the rest could not have moved the l2 score (the v2/l2 mismatch, quantified)")
else:
    print("skipped (no QDRANT_CLOUD_URL)")

l2 above-gold:   0%|          | 0/85 [00:00<?, ?it/s]

l2 above-gold:   1%|          | 1/85 [00:02<02:56,  2.10s/it]

l2 above-gold:   2%|▏         | 2/85 [00:04<02:47,  2.01s/it]

l2 above-gold:   4%|▎         | 3/85 [00:05<02:38,  1.93s/it]

l2 above-gold:   5%|▍         | 4/85 [00:07<02:33,  1.89s/it]

...

l2 above-gold:  98%|█████████▊| 83/85 [02:36<00:04,  2.30s/it]

l2 above-gold:  99%|█████████▉| 84/85 [02:38<00:02,  2.10s/it]

l2 above-gold: 100%|██████████| 85/85 [02:39<00:00,  1.97s/it]

l2 above-gold: 100%|██████████| 85/85 [02:39<00:00,  1.88s/it]

sub-1.0 ties: 85 | l2 above-gold docs: 141 (already judged: 101, MISSED: 40)
of 371 judged pairs, only 252 land in some l2 top-k -> the rest could not have moved the l2 score (the v2/l2 mismatch, quantified)


In [17]:
# Build the pairs we MISSED: l2 above-gold docs not yet judged.
if QDRANT:
    rows = []
    qtext = dict(zip(zip(sub1["dataset"], sub1["query_id"]), sub1["query"]))
    for (ds, qid), above in l2_above.items():
        todo = {d for d in above if (ds, qid, d) not in judged}
        if not todo:
            continue
        texts = s.corpus_text(ds, todo)
        for doc_id in todo:
            text = texts.get(doc_id)
            if text:
                rows.append({"dataset": ds, "query_id": qid, "doc_id": doc_id,
                             "query": qtext[(ds, qid)], "doc_text": text})
    new_pairs = pd.DataFrame(rows)
    print(f"new l2-above-gold pairs to judge: {len(new_pairs)}")
    new_pairs.head()
else:
    print("skipped")

new l2-above-gold pairs to judge: 40


In [18]:
# Judge the missed pairs (idempotent — skips the 371 already banked). Needs SPEND + a passed gate.
if QDRANT and SPEND and len(new_pairs):
    runs = JudgeRunLog(config).load()
    passed = runs[runs["passed"]] if not runs.empty else runs
    run_id = str(passed.iloc[-1]["judge_run_id"]) if not passed.empty else None
    counts = (RelevanceJudge(config).judge_pairs(new_pairs, run_id=run_id, budget=budget())
              if run_id else {"error": "no passed validation run"})
    counts
elif QDRANT:
    print(f"set SPEND=True to judge {len(new_pairs)} missed pairs (~a few cents)")
else:
    print("skipped")

set SPEND=True to judge 40 missed pairs (~a few cents)


In [19]:
# TRUE tie-conversion: l2 rankings (cached) scored before (human) vs after (human + ALL judged now).
if QDRANT:
    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str, "relevance": int})
    datasets = sorted(atoms["dataset"].unique())
    human = PilotScorer(config)._human_store(datasets)
    merged = QrelStore.concat([human, RelevanceJudge(config).as_qrelstore()])
    obj = RouterObjective(min_relevance=config.min_relevance)

    conv = []
    for (ds, qid), rk in l2_cache.items():
        gb = human.lookup(ds).get(qid, {})
        ga = merged.lookup(ds).get(qid, {})
        sb = {r: obj.assess(rank, gb)[0] for r, rank in rk.items()}
        sa = {r: obj.assess(rank, ga)[0] for r, rank in rk.items()}
        conv.append({"regime_before": regime(sb), "regime_after": regime(sa),
                     "resolved": regime(sb) == "all_tied" and regime(sa) != "all_tied"})
    conv = pd.DataFrame(conv)
    tied = int((conv["regime_before"] == "all_tied").sum())
    broke = int(conv["resolved"].sum())
    print(f"rows: {len(conv)} | all_tied before: {tied} | broke after judged gold: {broke}")
    print("TRUE l2 tie_conversion (l2 above-gold docs judged):",
          round(broke / tied, 3) if tied else "n/a")
    display(conv.groupby(["regime_before", "regime_after"]).size().rename("rows"))
else:
    print("skipped")

rows: 85 | all_tied before: 81 | broke after judged gold: 12
TRUE l2 tie_conversion (l2 above-gold docs judged): 0.148


regime_before  regime_after
all_tied       all_tied        69
               low_margin      12
low_margin     low_margin       4
Name: rows, dtype: int64

## Next steps

- **Not built (gated):** the §3a formal 500–1K sample-test router-population arm (placebo +
  ≥5 seeds) and Phase 2 dataset-wide deepening (~72K rows, ~$390) — both wait on the SPEC
  decision *"may the route label be defined below current truth depth?"*.
- **Calibration:** luna prices in `config.py` are placeholders — set them from luna's real card.
- **CLI twin** for headless runs: `poetry run python src/scripts/run_relevance_judge.py --audit | --validate | --pilot-sub1 | --score`.